In [2]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *

# Set absolute path for new metastore
new_metastore_path = os.path.abspath('/apps/sandbox/metastore')
metastore_url = f"jdbc:derby:;databaseName={new_metastore_path};create=true"

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    .set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .set("spark.sql.catalogImplementation", "hive")
    .set("spark.sql.catalog.spark_catalog.warehouse","s3a://warehouse/default")
    .set("spark.sql.warehouse.dir", "s3a://warehouse/default/spark")
    .set("javax.jdo.option.ConnectionURL", metastore_url)
    .set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.797,io.delta:delta-spark_2.12:3.3.2")
    .set("spark.executor.heartbeatInterval", "300000")
    .set("spark.network.timeout", "400000")
    .set("spark.hadoop.fs.defaultFS", "s3a://defaultfs/")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    .set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    .set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.delta.enableFastS3AListFrom", "true")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

# configure the SparkSession with the configure_spark_with_delta_pip() utility function in Delta Lake:
builder = SparkSession.builder.appName("00-deltalake-catalogs").master("local[*]").config(conf=sparkConf)
spark = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4"]).getOrCreate()

#
spark.sparkContext.setLogLevel('ERROR')
spark

#
# 
#
def display(df):
    df.show(truncate=False)

:: loading settings :: url = jar:file:/opt/spark-3.5.3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/brijeshdhaker/.ivy2/cache
The jars for the packages stored in: /home/brijeshdhaker/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b7f1455f-87f4-49bc-8656-749bde34f2fa;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in local-m2-cache
	found io.delta#delta-storage;3.3.2 in local-m2-cache
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
	found org.apache.hadoop#hadoop-aws;3.3.4 in local-m2-cache
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in local-m2-cache
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in local-m2-cache
:: resolution report :: resolve 185ms :: artifacts dl 9ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from local-m2-cache in [default]
	io.delta#delta-spark_2.12;3.3.2 from local-m2-cache in [default]
	io.delta#delta-storage;3.3.2 from local-m2-cache in [default

In [3]:
import os

#
jdbcHostname = "mysqlserver.sandbox.net"
jdbcDatabase = "SANDBOXDB"
jdbcPort = 3306
db_user = os.getenv('MYSQL_ADMIN_USER')
db_passwd = os.getenv('MYSQL_ADMIN_PASSWORD')

jdbcUrl = "jdbc:mysql://{0}:{1}/{2}?user={3}&password={4}".format(
    jdbcHostname,
    jdbcPort,
    jdbcDatabase,
    db_user,
    db_passwd
)

# print(jdbcUrl)

In [4]:
#
orders_query = "(select * from ORDERS) orders_alias"

#
orders_df = spark.read.jdbc(url=jdbcUrl, table=orders_query)

#
orders_df.show()

+----------+----------+------------+
|PRODUCT_ID|ORDER_DATE|ORDER_AMOUNT|
+----------+----------+------------+
|         1|2025-11-01|     19.5000|
|         1|2025-11-02|     28.5000|
|         1|2025-11-03|     48.5000|
|         1|2025-11-04|     16.5000|
|         1|2025-11-05|     29.5000|
|         1|2025-11-06|     36.5000|
|         1|2025-11-07|     28.5000|
|         1|2025-11-08|     47.6500|
|         1|2025-11-09|     58.5000|
|         1|2025-11-10|     28.5000|
|         1|2025-11-11|     10.5000|
|         1|2025-11-12|     68.5000|
|         1|2025-11-13|     38.5000|
|         1|2025-11-14|     58.5000|
|         1|2025-11-15|     28.5000|
|         1|2025-11-16|     50.5000|
|         1|2025-11-17|     30.0000|
|         1|2025-11-18|     88.0000|
|         1|2025-11-19|     35.0000|
|         1|2025-11-20|     11.0000|
+----------+----------+------------+
only showing top 20 rows



In [5]:
#
products_query = "(select * from PRODUCTS) products_alias"

#
products_df = spark.read.jdbc(url=jdbcUrl, table=products_query)

#
products_df.show()

+----------+--------------+
|PRODUCT_ID|  PRODUCT_NAME|
+----------+--------------+
|         1|        Mobile|
|         2|    Television|
|         3|Air Conditions|
|         4|          Fans|
+----------+--------------+



In [1]:
import os
print(os.getenv('JAVA_HOME'))

/usr/lib/jvm/java-1.17.0-openjdk-amd64
